# Data Explorer: DABStep Benchmark Inference

This notebook demonstrates the Data Explorer agent solving questions on the [DABStep benchmark](https://huggingface.co/datasets/adyen/DABstep) (Data Agent Benchmark for Multi-step Reasoning) **from scratch** -- the agent receives only the data files and a question, and uses Python (pandas, numpy) via a tool-calling loop to explore, reason, and answer.

The agent has no pre-distilled domain helpers and no few-shot examples -- it figures out everything itself. This shows the bare agentic workflow; for production-quality answers you'd run the full learn -> distill -> inference pipeline (see `dabstep_agent/learn/`).


## The Three-Phase Architecture

```
Phase 1: LEARNING                Phase 2: INFERENCE              Phase 3: REFLECTION
┌─────────────────┐             ┌─────────────────┐             ┌─────────────────┐
│ Heavy Model      │             │ Light Model      │             │ Heavy Model      │
│ (Opus 4.5/4.6)  │             │ (Claude Haiku)   │             │ (Opus/Sonnet)   │
│                  │             │                  │             │                  │
│ Multi-pass loop  │   distill   │ Single-pass loop │   review    │ Reflection +     │
│ + ground truth   │───────────▶│ + helper.py      │───────────▶│ Self-consistency │
│ + full toolkit   │             │ + few-shot only  │             │ (offline)        │
└─────────────────┘             └─────────────────┘             └─────────────────┘
     ▼ output:                       ▼ output:                       ▼ output:
  helper.py                       answer + trace                  improved prompts
  solutions.md                                                    fed back to Phase 2
```

**This launchable demonstrates Phase 2 with no pre-distilled artifacts** -- the agent receives the data files and a question, and solves from scratch. To see Phase 2 with distilled helpers, run `dabstep_agent/learn/learn.py` and `dabstep_agent/learn/distill_nat/distill.py` first to populate `helper.py` and `new_solutions.md`.


## Setup

The DABStep inference server should already be running on port 8000. Let's verify the connection and check the data.

In [ ]:
import os
import sys
import json
import shutil
import subprocess
import pathlib
import requests
import time
from dotenv import load_dotenv

# Find repo root (works on both Brev and Docker)
repo_root = None
_result = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True)
if _result.returncode == 0:
    repo_root = _result.stdout.strip()
if not repo_root:
    _p = pathlib.Path.cwd()
    for _dir in [_p] + list(_p.parents):
        if (_dir / 'pyproject.toml').exists():
            repo_root = str(_dir)
            break
if not repo_root:
    for _c in [pathlib.Path.home() / 'data-explorer-launchable', pathlib.Path('/app')]:
        if (_c / 'pyproject.toml').exists():
            repo_root = str(_c)
            break
if not repo_root:
    raise RuntimeError('Could not find repo root')

os.chdir(repo_root)
for _path in [repo_root, os.path.join(repo_root, 'src')]:
    if _path not in sys.path:
        sys.path.insert(0, _path)

# Install project + dependencies into the running kernel (instant if already done)
uv_bin = shutil.which('uv') or os.path.expanduser('~/.local/bin/uv')
_r = subprocess.run(
    [uv_bin, 'pip', 'install', '--quiet', '--prerelease', 'allow',
     '--python', sys.executable, '-e', '.'],
    cwd=repo_root, capture_output=True, text=True)
if _r.returncode != 0:
    print(f'WARNING: uv pip install failed:\n{_r.stderr}')

load_dotenv('secrets.env')

SERVER_URL = 'http://localhost:8000'

assert os.environ.get('ANTHROPIC_API_KEY'), (
    'ANTHROPIC_API_KEY not set. Get a key at https://console.anthropic.com '
    'and set it: export ANTHROPIC_API_KEY=sk-ant-your_key_here'
)

# Wait for server to be ready (it starts in the background)
for attempt in range(10):
    try:
        resp = requests.get(f'{SERVER_URL}/docs', timeout=5)
        if resp.status_code == 200:
            print(f'DABStep server is ready at {SERVER_URL}')
            break
    except requests.ConnectionError:
        if attempt < 9:
            print(f'Waiting for server... (attempt {attempt + 1}/10)')
            time.sleep(5)
        else:
            print('Server not responding. Start it manually with:')
            print('  uv run python dabstep_agent/inference/server.py &')

## Explore the DABStep Dataset

DABStep is a financial payments dataset with 450 tasks focused on merchant fees, transactions, and domain-specific rules. Let's peek at the data.

In [ ]:
import pandas as pd

payments = pd.read_csv(os.path.join(repo_root, 'data/context/payments.csv'))
print(f'Payments: {len(payments):,} transactions')
print(f'Columns: {", ".join(payments.columns)}')
print()
payments.head(3)

In [ ]:
with open(os.path.join(repo_root, 'data/context/fees.json')) as f:
    fees = json.load(f)
print(f'Fee rules: {len(fees)}')
print(f'Sample rule keys: {", ".join(fees[0].keys())}')
print()
print('First fee rule:')
json.dumps(fees[0], indent=2)

In [ ]:
# Load sample questions from the dev set
tasks_path = os.path.join(repo_root, 'data/tasks_dev.json')
if os.path.exists(tasks_path):
    with open(tasks_path) as f:
        tasks = json.load(f)
    print(f'Dev tasks: {len(tasks)}')
    print()
    for i, t in enumerate(tasks[:5]):
        print(f'Task {t.get("task_id", i)}: {t["question"][:100]}...')
        print(f'  Answer: {t.get("answer", "N/A")}')
        print()
else:
    print(f'Dev tasks not found at {tasks_path}.')
    print('Run: uv run python dabstep_agent/download_data.py')

## Ask the Agent a Question

Let's send a question to the running DABStep inference server. The agent (Claude Haiku) has access to the data files via a Python executor tool -- it explores the data, writes code, and iterates to find an answer. No pre-distilled domain knowledge: pure tool-calling loop from scratch.


In [ ]:
def ask_dabstep(question: str, guidelines: str = 'N/A') -> dict:
    """Send a question to the DABStep inference server."""
    t0 = time.time()
    resp = requests.post(
        f'{SERVER_URL}/solve',
        json={'question': question, 'guidelines': guidelines},
        timeout=300
    )
    elapsed = time.time() - t0
    result = resp.json()
    result['elapsed_seconds'] = round(elapsed, 1)
    return result


# Example: a straightforward question
result = ask_dabstep(
    question='Which issuing country has the highest number of transactions?'
)

print(f'Answer: {result["agent_answer"]}')
print(f'Time:   {result["elapsed_seconds"]}s')
print(f'Trace:  {len(result["reasoning_trace"])} messages')

In [ ]:
# View the reasoning trace (what the agent did step by step)
for msg in result['reasoning_trace']:
    role = msg.get('role', '?')
    content = msg.get('content', '')
    tool = msg.get('tool_name', '')

    if role == 'human':
        print(f'[PROMPT] {content[:200]}...')
    elif role == 'ai':
        if msg.get('tool_calls'):
            for tc in msg['tool_calls']:
                code = tc.get('args', {}).get('code', '')[:200]
                print(f'[AGENT] Tool call: {tc.get("name", "?")}  →  {code}...')
        elif content:
            print(f'[AGENT] {content[:300]}')
    elif role == 'tool':
        print(f'[TOOL:{tool}] {content[:200]}')
    print()

## More Examples

Try some harder questions that require multi-step reasoning, fee calculations, and cross-referencing multiple data sources.

In [ ]:
hard_questions = [
    ('What is the average transaction value grouped by shopper_interaction for '
     "Crossfit_Hanna's TransactPlus transactions between January and April 2023?",
     "['POS: 89.34', 'Ecommerce: 92.70']"),
    ('What is the fee ID or IDs that apply to account_type = H and aci = D?',
     '5, 9, 20, 30, 31, 41, 46, 48, ... (161 IDs)'),
    ('For the 200th of the year 2023, what is the total fees (in euros) that Rafa_AI should pay?',
     '52.37'),
]

for q, expected in hard_questions:
    print(f'Q: {q}')
    r = ask_dabstep(q)
    print(f'A: {r["agent_answer"]}  ({r["elapsed_seconds"]}s)')
    print(f'Expected: {expected}')
    print('-' * 60)

## Try Your Own Questions

The DABStep dataset covers financial payments: merchants, fees, transactions, card schemes, and fraud. Ask anything about the data below.

In [ ]:
# Type your own question here:
# result = ask_dabstep('Your question about the DABStep financial dataset here')
# print(f'Answer: {result["agent_answer"]}')